# FEMA NFHL Pipeline Demo

This notebook demonstrates the package workflow for a small case study: Alameda County, California. Core logic lives in `src/fema_nfhl`; the notebook only orchestrates a few clean outputs for review.


## Scientific Guardrails

- FEMA NFHL is treated as authoritative public flood hazard mapping data, not as an official determination produced by this tool.
- This workflow focuses on ingestion, validation, transformation, web mapping, and floodplain area summaries.
- Derived flood-depth modeling is intentionally excluded because BFE-to-DEM surfaces require stronger vertical datum, hydraulic connectivity, and interpolation assumptions than this project should claim.
- Area calculations should use a projected or equal-area CRS, not web-map coordinates.


In [ ]:
from pathlib import Path

import pandas as pd
import yaml

from fema_nfhl.catalog import catalog_extracted_layers
from fema_nfhl.case_study import prepare_county_case_study
from fema_nfhl.exposure import floodplain_area_summary
from fema_nfhl.mapping import create_interactive_map
from fema_nfhl.validate import validate_layers
from fema_nfhl.visualize import create_exposure_bar_chart

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
config = yaml.safe_load((PROJECT_ROOT / 'config' / 'example_config.yaml').read_text(encoding='utf-8'))
config


## Paths

Update these paths after downloading and extracting a small NFHL case-study package. The optional boundary file is only needed if you want to clip a larger extracted folder to Alameda County before mapping or overlay.


In [ ]:
extracted_nfhl = PROJECT_ROOT / 'data' / 'extracted'
case_study_nfhl = PROJECT_ROOT / 'data' / 'case_study' / 'alameda'
admin_boundaries = PROJECT_ROOT / 'data' / 'boundaries' / 'alameda_county.shp'
outputs = PROJECT_ROOT / 'outputs'
outputs.mkdir(parents=True, exist_ok=True)

extracted_nfhl, case_study_nfhl, admin_boundaries


## Prepare Alameda County Case Study

If you already extracted only the Alameda NFHL package, you can skip clipping and point `case_study_nfhl` directly to that extracted folder. If you have a larger extracted input plus a county boundary, this cell clips common NFHL layers to Alameda County.


In [ ]:
if extracted_nfhl.exists() and admin_boundaries.exists():
    prepare_county_case_study(
        input_path=extracted_nfhl,
        output_dir=case_study_nfhl,
        boundary_path=admin_boundaries,
        boundary_id_field=config['case_study']['boundary_id_field'],
        boundary_id_value=config['case_study']['boundary_id_value'],
    )
else:
    print('Skip clipping until extracted NFHL data and an Alameda boundary file are available.')


## Catalog Extracted NFHL Layers

The catalog lists available NFHL layers, geometry types, CRS, feature counts, and key FEMA attributes.


In [ ]:
if case_study_nfhl.exists():
    catalog_csv = outputs / 'nfhl_catalog.csv'
    catalog_extracted_layers(case_study_nfhl, catalog_csv)
    display(pd.read_csv(catalog_csv).head(10))
else:
    print('Case-study folder not found yet.')


## Validate Layers

The validation report checks required NFHL layers, CRS presence, geometry validity, empty geometries, flood-zone attributes, and BFE attribute quality where the `S_BFE` layer is available.


In [ ]:
if case_study_nfhl.exists():
    validation_csv = outputs / 'validation_report.csv'
    validate_layers(case_study_nfhl, output_csv=validation_csv)
    display(pd.read_csv(validation_csv).head(20))
else:
    print('Case-study folder not found yet.')


## Create Interactive Map

The map uses an EPSG:4326 copy for web display only. Analytical CRS handling stays inside the package functions.


In [ ]:
if case_study_nfhl.exists():
    map_path = outputs / 'flood_hazard_map.html'
    create_interactive_map(case_study_nfhl, map_path)
    print(map_path)
else:
    print('Case-study folder not found yet.')


## Floodplain Area Summary

This overlay calculates flood hazard area by administrative unit and FEMA flood zone after reprojection to an equal-area CRS.


In [ ]:
flood_layer = next(case_study_nfhl.rglob('S_FLD_HAZ_AR.shp'), None) if case_study_nfhl.exists() else None
if flood_layer and admin_boundaries.exists():
    exposure_csv = outputs / 'floodplain_area_summary.csv'
    floodplain_area_summary(
        flood_layer=flood_layer,
        admin_boundaries=admin_boundaries,
        output_csv=exposure_csv,
        admin_id_field=config['case_study']['boundary_id_field'],
        admin_name_field=config['case_study']['boundary_name_field'],
        equal_area_crs=config['analysis']['equal_area_crs'],
    )
    display(pd.read_csv(exposure_csv).head(20))
else:
    print('Skip exposure summary until S_FLD_HAZ_AR and admin boundaries are available.')


## Plot Exposure Summary

The chart is a lightweight portfolio visual for the CSV exposure summary.


In [ ]:
exposure_csv = outputs / 'floodplain_area_summary.csv'
if exposure_csv.exists():
    chart_path = outputs / 'floodplain_area_summary.png'
    create_exposure_bar_chart(exposure_csv, chart_path, title='Alameda County Floodplain Area By FEMA Zone')
    display(chart_path)
else:
    print('Exposure CSV not found yet.')
